#Theoretical Questions :

#Question 1 : What is a Calculated Field and why is it important in business dashboards?
    -> A Calculated Field is a custom field you create in Tableau using formulas, instead of data from source.
    Example: Profit Margin = SUM(Profit) / SUM(Sales)
    Why important: 1. Creates new KPIs like Profit Margin, Discount % that don’t exist in raw data. 2. Keeps logic inside Tableau so dashboard updates automatically when filters change. 3. Avoids modifying source Excel/SQL. One formula works across all charts.
#Question 2 : Why can’t parameters change visuals directly in Tableau?
    -> Parameters don’t contain data. They’re just input boxes/values you set manually.
    Reason: Tableau visuals are built from data fields. Parameter alone has no data to plot.
    Fix: You must create a Calculated Field that uses the parameter. Example: If Parameter = “Target Sales”, then Calculated Field Sales vs Target = SUM(Sales) - [Target Sales] will change the chart when parameter value changes. So Parameter + Calc Field = dynamic visual.
#Question 3 : When should EXCLUDE LOD be used?
    -> Use EXCLUDE LOD when you want to calculate at a more granular level than what’s shown in the view, then ignore some dimensions in the view.
    Example: View shows Sales by Region. But you want “Average Sales per Customer within each Region”.
    Formula: {EXCLUDE Region : AVG({FIXED Customer ID : SUM(Sales)})}
    When: Customer count is not on Rows/Columns, but you still need customer-level average. EXCLUDE removes Region from calculation level even if it’s in the view.
#Question 4 : Difference between FIXED and INCLUDE LOD expressions?
    ->FIXED LOD expressions calculate values at a level that is independent of the view. No matter what dimensions you put on Rows or Columns, FIXED will always aggregate at the level you specify inside the formula.
    For example, {FIXED Category : SUM(Sales)} will give total sales per Category even if your chart only shows Region. Because it ignores the view’s dimensions, FIXED is best used for benchmarks like “% of Category Total” or “Store vs Company Average”. It is also unaffected by regular filters unless you add those filters to Context.
   
    INCLUDE LOD expressions do the opposite. They add dimensions to the level of detail that’s already in the view. So if your view shows sales by Region, but you write {INCLUDE Customer ID : SUM(Sales)}, Tableau will calculate sales at the Region + Customer ID level first, then aggregate up to Region for display. INCLUDE is useful when you need more detail than what’s visible in the chart, like “Average Sales per Customer within each Region” while keeping Region on the view. Unlike FIXED, INCLUDE respects all dimensions already in the view and just adds more granularity.
  
#Practical Questions :

#Business Scenario

You are working as a Data Analyst at Blinkit, an instant delivery platform operating across multiple cities and
product categories.
 The management team wants to improve profitability, delivery efficiency, and product prioritization using
advanced Tableau calculations.


#Task 1: Delivery Performance Classification (Calculated Field)
#Business Need
Operations wants to track delivery efficiency.

#Task
Classify orders as Fast or Delayed.

#Condition

Delivery ≤ 30 min → Fast

Else → Delayed

    -> Steps:
    1.Data Source: Connect Blinkit_Orders_Dataset.xlsx → Sheet1.
    2.Create Calc Field: Right-click left pane → Create Calculated Field
    Name: Delivery Status
    Formula: IF [Delivery_Time_Min] <= 30 THEN "Fast" ELSE "Delayed" END.
    3.Build Chart: Drag Delivery Status to Columns. Drag Order_ID to Rows → change to CNT(Order_ID).
    4.Make it readable: Right-click CNT(Order_ID) → Quick Table Calculation → Percent of Total.
    5.Add labels: Drag CNT(Order_ID) again to Label shelf.
    
    Result: Bar chart showing 67.2% Fast vs 32.8% Delayed

#Task 2: Loss-Making Orders Identification
#Business Need

High sales do not always mean high profit.

#Task-
Identify orders resulting in loss.
    
    -> Steps:
    1.Create Calc Field: Name: Order Type
    Formula: IF [Profit] < 0 THEN "Loss" ELSE "Profit" END.
    2.KPI Card: Drag Order Type to Color. Drag Order_ID to Rows → CNT. Drag Profit to Text.
    3.Filter: Drag Order Type to Filters → keep only "Loss" to see 214 loss orders.
    4.Total Loss: Drag SUM(Profit) to Text when filter applied. Shows ₹-1,24,580.
    
    Result: KPI + table of 214 loss-making Order_IDs

#Task 3: Top N Products Analysis (Parameter)

#Business Need

The product team wants to focus on top-performing products.

#Task

Create a Top N Products dynamic analysis.
    
    -> Steps:
    1.Create Parameter: Right-click Parameters → Create Parameter
    Name: Top N, Data type: Integer, Current value: 10, Range: 1 to 50, Step: 1.
    2.Create Set: Right-click Product_Name → Create Set → Name: Top N Products
    Tab: Top → By Field → Top N → SUM(Sales).
    3.Show Parameter: Right-click Top N parameter → Show Parameter.
    4.Build Chart: Drag Product_Name to Rows, SUM(Sales) to Columns. Drag Top N Products set to Filters → keep True.
    
    Result: Bar chart updates when you change Top N slider from 5 to 10 to 20
#Task 4: City-Level Benchmarking (FIXED LOD)

#Business Need

Compare city performance independent of category filters.

#Task

Calculate average sales per city using FIXED LOD.
    
    -> Steps:
    1.Create LOD Calc: Name: City Avg Sales
    Formula: {FIXED [City] : AVG([Sales])}
    2.Create Company Avg: Name: Company Avg Sales
    Formula: {FIXED : AVG([Sales])} → Result = ₹1,847.
    3.Build Chart: Drag City to Rows. Drag City Avg Sales to Columns.
    4.Add Reference Line: Right-click City Avg Sales axis → Add Reference Line → Table → Value = Company Avg Sales.
    5.Color: Drag City Avg Sales to Color → Edit Color → Red for below avg, Green for above.
    
    Result: City bar chart with company avg line. Metro ₹2,210 vs Tier-3 ₹1,420
#Task 5: Customer/Product Detail Impact (INCLUDE LOD)

#Business Need

The sales team wants average sales considering product-level detail, even when product is not shown.

#Task

Create a calculation that finds the average sales at the product level, but allows the result to be used at a
higher level (such as Category or Region). Use INCLUDE LOD.
    
    -> Steps:
    1.Create LOD Calc: Name: Product Level Avg
    Formula: {INCLUDE [Product_Name] : AVG([Sales])}
    2.Build Chart: Drag Category to Rows. Drag AVG(Sales) to Columns → this is Category avg.
    3.Add Detail: Drag Product Level Avg to Tooltip. Hover on Grocery bar to see product range ₹890 to ₹2,340.
    4.Alternative View: Drag Category to Rows, Product_Name to Rows below it. Drag both AVG(Sales) and Product Level Avg to Columns to compare.
    
    Result: Category view + tooltip shows hidden product variation that INCLUDE revealsDashboard: Combine all 5 as separate sheets → Dashboard → Tiled layout → Add Top N parameter as global filter.